# Anomaly Vectorization Comparison

This notebook compares two anomaly feature spaces:

- Sparse character TF-IDF
- Dense TF-IDF + LSA

Goal:

- Check if LSA helps anomaly ranking quality and stability.
- Compare overlap between top anomaly sets.
- Export side-by-side diagnostics for report discussion.

Reader guide:

- Both models use the same normalized input text.
- Only the vector representation changes.
- Ranking outputs are saved to `data/results/`.

In [9]:
from pathlib import Path

import numpy as np
import pandas as pd
from scipy.sparse import issparse
from scipy.stats import spearmanr

from anomaly_detection import TextAnomalyDetector
from core.data_io import ArticleDataset
from exploration import build_anomaly_candidate_table
from preprocessing import StructuralFeatureExtractor, TextNormalizer, TextPreprocessor


ImportError: cannot import name 'StructuralFeatureExtractor' from 'preprocessing' (/home/samuel/Documents/data-mining-anomaly-detection/src/preprocessing/__init__.py)

## Load and normalize data


In [ ]:
project_root_path = Path.cwd().parent
results_data_directory_path = project_root_path / "data" / "results"
results_data_directory_path.mkdir(parents=True, exist_ok=True)

articles_csv_path = project_root_path / "data" / "raw" / "articles.csv"
articles_data_frame = ArticleDataset(input_csv_path=articles_csv_path).load_articles()

document_ids = articles_data_frame["doc_id"].tolist()
raw_texts = articles_data_frame["text"].tolist()
normalized_bundle = TextNormalizer().normalize_for_both_tasks(raw_texts)

structural_feature_extractor = StructuralFeatureExtractor()
structural_feature_matrix = structural_feature_extractor.transform(raw_texts)
structural_feature_matrix.shape


## Model A: Sparse TF-IDF (char n-grams) + structural features

### This model keeps a sparse lexical view and adds dense structural signals.


In [ ]:
sparse_preprocessor = TextPreprocessor(
    vectorization_model_name="tfidf",
    max_features=30000,
    min_document_frequency=1,
    max_document_frequency=1.0,
    ngram_range=(3, 5),
    analyzer_mode="char_wb",
)

sparse_features = sparse_preprocessor.fit_transform(normalized_bundle.anomaly_texts)
if issparse(sparse_features):
    sparse_dense_features = np.asarray(sparse_features.todense(), dtype=np.float64)
else:
    sparse_dense_features = np.asarray(sparse_features, dtype=np.float64)

sparse_hybrid_features = np.hstack([sparse_dense_features, structural_feature_matrix])

sparse_detector = TextAnomalyDetector(contamination_ratio=0.02, random_seed=42)
sparse_mask, sparse_scores = sparse_detector.run_detection(sparse_hybrid_features)

sparse_table = build_anomaly_candidate_table(
    document_ids=document_ids,
    anomaly_scores=sparse_scores.tolist(),
    anomaly_mask=sparse_mask.tolist(),
)
sparse_table.to_csv(results_data_directory_path / "notebook_07_sparse_hybrid_anomaly_candidates.csv", index=False)

sparse_table.head(20)


## Model B: Dense TF-IDF + LSA + structural features
#
# This model uses dense semantic compression and the same structural signals.


In [ ]:
lsa_preprocessor = TextPreprocessor(
    vectorization_model_name="tfidf_lsa_dense",
    max_features=30000,
    min_document_frequency=1,
    max_document_frequency=1.0,
    ngram_range=(3, 5),
    analyzer_mode="char_wb",
    dense_embedding_dimension=256,
    random_seed=42,
)

lsa_features = lsa_preprocessor.fit_transform(normalized_bundle.anomaly_texts)
lsa_dense_features = np.asarray(lsa_features, dtype=np.float64)
lsa_hybrid_features = np.hstack([lsa_dense_features, structural_feature_matrix])

lsa_detector = TextAnomalyDetector(contamination_ratio=0.02, random_seed=42)
lsa_mask, lsa_scores = lsa_detector.run_detection(lsa_hybrid_features)

lsa_table = build_anomaly_candidate_table(
    document_ids=document_ids,
    anomaly_scores=lsa_scores.tolist(),
    anomaly_mask=lsa_mask.tolist(),
)
lsa_table.to_csv(results_data_directory_path / "notebook_07_lsa_hybrid_anomaly_candidates.csv", index=False)

lsa_table.head(20)


## Compare top anomalies (top-20 side by side)


In [ ]:
comparison_table = pd.DataFrame(
    {
        "rank": range(1, 21),
        "sparse_doc_id": sparse_table.head(20)["doc_id"].tolist(),
        "lsa_doc_id": lsa_table.head(20)["doc_id"].tolist(),
        "sparse_score": sparse_table.head(20)["score"].tolist(),
        "lsa_score": lsa_table.head(20)["score"].tolist(),
    }
)
comparison_table.to_csv(results_data_directory_path / "notebook_07_top20_comparison.csv", index=False)
comparison_table


## Compare overlap diagnostics


In [ ]:
def compute_top_k_overlap(
    sparse_ranked_table: pd.DataFrame,
    lsa_ranked_table: pd.DataFrame,
    top_k: int,
) -> int:
    """Calculates overlap between two ranked top-k anomaly sets."""
    sparse_top_ids = set(sparse_ranked_table.head(top_k)["doc_id"].tolist())
    lsa_top_ids = set(lsa_ranked_table.head(top_k)["doc_id"].tolist())
    return len(sparse_top_ids.intersection(lsa_top_ids))


overlap_diagnostics = pd.DataFrame(
    {
        "top_k": [10, 25, 50, 75, 100],
        "overlap_count": [
            compute_top_k_overlap(sparse_table, lsa_table, top_k_value) for top_k_value in [10, 25, 50, 75, 100]
        ],
    }
)
overlap_diagnostics.to_csv(results_data_directory_path / "notebook_07_overlap_diagnostics.csv", index=False)
overlap_diagnostics


## Why overlap can be low


In [ ]:
spearman_result = spearmanr(sparse_scores, lsa_scores)
score_rank_correlation = float(spearman_result.statistic)

summary_diagnostics = {
    "sparse_predicted_anomaly_count": int(sparse_mask.sum()),
    "lsa_predicted_anomaly_count": int(lsa_mask.sum()),
    "spearman_score_correlation": float(score_rank_correlation),
}
summary_diagnostics_data_frame = pd.DataFrame([summary_diagnostics])
summary_diagnostics_data_frame.to_csv(
    results_data_directory_path / "notebook_07_summary_diagnostics.csv",
    index=False,
)
summary_diagnostics


## Overlap score in top-50


In [ ]:
sparse_top_50 = set(sparse_table.head(50)["doc_id"].tolist())
lsa_top_50 = set(lsa_table.head(50)["doc_id"].tolist())
overlap_count = len(sparse_top_50.intersection(lsa_top_50))

pd.DataFrame([{"top_50_overlap_count": overlap_count}]).to_csv(
    results_data_directory_path / "notebook_07_top50_overlap.csv",
    index=False,
)
overlap_count


## Deterministic top-50 export candidate
#
# This export uses score ranking from the dense hybrid model.
# It gives exactly 50 ids for assignment-style output.


In [ ]:
lsa_top_50_candidate = lsa_table.head(50).copy()
lsa_top_50_candidate["anomaly"] = lsa_top_50_candidate.index + 1

anomaly_candidate_output = lsa_top_50_candidate[["anomaly", "doc_id"]]
anomaly_candidate_output.to_csv(results_data_directory_path / "notebook_07_anomalies_candidate.csv", index=False)

anomaly_candidate_output.head(10)


### Files produced by this notebook
#
# - `data/results/notebook_07_sparse_hybrid_anomaly_candidates.csv`
# - `data/results/notebook_07_lsa_hybrid_anomaly_candidates.csv`
# - `data/results/notebook_07_top20_comparison.csv`
# - `data/results/notebook_07_overlap_diagnostics.csv`
# - `data/results/notebook_07_summary_diagnostics.csv`
# - `data/results/notebook_07_top50_overlap.csv`
# - `data/results/notebook_07_anomalies_candidate.csv`
